# Stage 5

In [61]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datetime import date, timedelta
import random
import string

import matplotlib.pyplot as plt

## Dataset

In [2]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
BATCH_SIZE = 64
SEED = 42

# no peaking into the future !!
START_DATE = date(1990, 1, 1)
END_DATE = date(2030, 12, 31)
TEST_HOLDOUT = 1000
ALL_DATES = [START_DATE + timedelta(days=i)
             for i in range((END_DATE - START_DATE).days + 1)]
random.Random(SEED).shuffle(ALL_DATES)
TRAIN_DATES = ALL_DATES[:-TEST_HOLDOUT]
TEST_DATES = ALL_DATES[-TEST_HOLDOUT:]

SOS = "<"
EOS = ">"
PAD = "_"
tokens = [SOS, EOS, PAD] + list(string.digits + string.ascii_letters + " /,-")
c2i = {c: i for i, c in enumerate(tokens)}
i2c = {i: c for c, i in c2i.items()}
VOCAB_SIZE = len(tokens)

MONTHS = ("January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December")
DATE_FORMATS = ("d/m/yyyy", "dd/mm/yyyy", "Month d, yyyy", "d Mon yyyy")


def render_date(d, format_idx):
    month = MONTHS[d.month - 1]
    return (
        f"{d.day}/{d.month}/{d.year}",
        f"{d.day:02d}/{d.month:02d}/{d.year}",
        f"{month} {d.day}, {d.year}",
        f"{d.day} {month[:3]} {d.year}",
    )[format_idx]


def encode(text):
    return torch.tensor([c2i[c] for c in text], dtype=torch.long)


def decode(ids):
    # Retains special tokens so you can inspect the exact sequence.
    return "".join(i2c[int(i)] for i in ids)

In [3]:
class DateDataset(Dataset):
    def __init__(self, test=False, epoch_size=4096):
        self.test = test
        self.epoch_size = epoch_size
        self.dates = TEST_DATES if test else TRAIN_DATES

    def __len__(self):
        if self.test:
            return len(self.dates) * len(DATE_FORMATS)
        return self.epoch_size

    def _prepare_one(self, d, format_idx):
        x = encode(render_date(d, format_idx))
        y = encode(SOS + d.isoformat() + EOS)
        return x, y

    def _generate_one(self):
        d = self.dates[torch.randint(len(self.dates), ()).item()]
        format_idx = torch.randint(len(DATE_FORMATS), ()).item()
        return self._prepare_one(d, format_idx)

    def __getitem__(self, idx):
        if self.test:
            date_idx, format_idx = divmod(idx, len(DATE_FORMATS))
            return self._prepare_one(self.dates[date_idx], format_idx)
        return self._generate_one()


def right_pad_seqs(batch):
    x, y = zip(*batch)
    x_lens = torch.tensor([len(s) for s in x])
    x_padded = pad_sequence(x, batch_first=True, padding_value=c2i[PAD])
    y_padded = pad_sequence(y, batch_first=True, padding_value=c2i[PAD])
    return x_padded, x_lens, y_padded


torch.manual_seed(SEED)
ds_train = DateDataset()
ds_test = DateDataset(test=True)
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, collate_fn=right_pad_seqs)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, collate_fn=right_pad_seqs)

# materialise 10 for repro sake
tiny_examples = [ds_train[i] for i in range(10)]
dl_tiny = DataLoader(tiny_examples, batch_size=10, collate_fn=right_pad_seqs)

In [13]:
[b[0].shape for b in tiny_examples], [b[1].shape for b in tiny_examples]

([torch.Size([11]),
  torch.Size([14]),
  torch.Size([11]),
  torch.Size([9]),
  torch.Size([10]),
  torch.Size([13]),
  torch.Size([16]),
  torch.Size([8]),
  torch.Size([11]),
  torch.Size([14])],
 [torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12]),
  torch.Size([12])])

In [5]:
for format_idx in range(len(DATE_FORMATS)):
    x, y = ds_train._prepare_one(date(2020, 3, 3), format_idx)
    print(f"{decode(x):20s} -> {decode(y)}")

x, x_lens, y = next(iter(dl_train))
print("Batch shapes:", x.shape, x_lens.shape, y.shape)
print("Source lengths:", x_lens[:5].tolist())
print("First padded source:", decode(x[0]))
print("First target:", decode(y[0]))

3/3/2020             -> <2020-03-03>
03/03/2020           -> <2020-03-03>
March 3, 2020        -> <2020-03-03>
3 Mar 2020           -> <2020-03-03>
Batch shapes: torch.Size([64, 17]) torch.Size([64]) torch.Size([64, 12])
Source lengths: [11, 10, 10, 11, 10]
First padded source: 21 Mar 2029______
First target: <2029-03-21>


## Batch contract → your Transformer


- Data stays on CPU until you move a batch to `DEVICE`.

For teacher forcing, the decoder input is `y[:, :-1]` and its labels are `y[:, 1:]`: 11 predictions, including EOS. The logits should have shape `(B, 11, VOCAB_SIZE)`.

Your attention needs source padding masking (both encoder self-attention and decoder cross-attention) and causal masking in decoder self-attention. There is no target padding in this particular task. Decide and document what a boolean `True` means in **your** mask API.

Start by memorizing `dl_tiny`. For held-out accuracy, generate from SOS without supplying the target, stop at EOS, and compare the whole ISO string. Teacher-forced token accuracy alone doesn't check free-running decoding.

## Model

Start with scaled dot-product attention and its reference check.

In [92]:
torch.tril(torch.ones((4, 4)))

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [ ]:
class SelfAttentionSingleHead(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.d = d
        self.K = nn.Linear(d, d, bias=False)
        self.Q = nn.Linear(d, d, bias=False)
        self.V = nn.Linear(d, d, bias=False)

    def forward(self, x, src_mask):
        # x is (B, T, d)

        keys = self.K(x)
        queries = self.Q(x)
        values = self.V(x)

        # key-query affinities
        e = queries @ torch.transpose(keys, -1, -2)
        e /= self.d**0.5  # "scaled" part
        e.masked_fill_(~src_mask.unsqueeze(1), -torch.inf)

        # attention weights
        alpha = torch.softmax(e, dim=-1)

        outputs = alpha @ values

        return outputs


class SelfAttentionMultiHead(nn.Module):
    def __init__(self, d: int, h, masked: bool = False):
        super().__init__()

        assert h > 0
        assert d % h == 0
        self.h = h
        self.d = d
        self.d_h = d // h
        self.masked = masked

        # assume d_h = d/h
        self.K = nn.Linear(d, d, bias=False)
        self.Q = nn.Linear(d, d, bias=False)
        self.V = nn.Linear(d, d, bias=False)
        self.W0 = nn.Linear(d, d, bias=False)

    def forward(self, x, src_mask):
        # x is (B, T, d)
        B, T, d = x.shape

        keys = self.K(x).view(B, T, self.h, self.d_h).transpose(1, 2)
        queries = self.Q(x).view(B, T, self.h, self.d_h).transpose(1, 2)
        values = self.V(x).view(B, T, self.h, self.d_h).transpose(1, 2)

        # key-query affinities
        e = queries @ torch.transpose(keys, -1, -2)  # (B, H, T, T)
        e /= self.d_h**0.5  # "scaled" part, now by d_h
        e.masked_fill_(~src_mask.unsqueeze(1).unsqueeze(1), -torch.inf)

        # causal/masking here!
        # make all "future" positions in the affinities be -inf
        if self.masked:
            causal_mask = (
                ~torch.tril(torch.ones((T, T), dtype=torch.bool))
                .unsqueeze(0)
                .unsqueeze(0)
            ).to(x.device)
            e.masked_fill_(causal_mask, -torch.inf)

        # attention weights
        alpha = torch.softmax(e, dim=-1)

        outputs = alpha @ values
        outputs.transpose_(1, 2)  # (B, H, T, d_h) -> (B, T, H, d_h)
        outputs = torch.reshape(outputs, (B, T, -1))
        outputs = self.W0(outputs)

        return outputs


class CrossAttentionMultiHead(nn.Module):
    def __init__(self, d: int, h):
        super().__init__()

        assert h > 0
        assert d % h == 0
        self.h = h
        self.d = d
        self.d_h = d // h

        # assume d_h = d/h
        self.K = nn.Linear(d, d, bias=False)
        self.Q = nn.Linear(d, d, bias=False)
        self.V = nn.Linear(d, d, bias=False)
        self.W0 = nn.Linear(d, d, bias=False)

    def forward(self, x, src_mask, x_enc):
        # x is (B, T, d)
        B, T, d = x.shape
        _, S, _ = x_enc.shape  # x_enc may (will) have diff seq lengths

        keys = self.K(x_enc).view(B, S, self.h, self.d_h).transpose(1, 2)
        queries = self.Q(x).view(B, T, self.h, self.d_h).transpose(1, 2)
        values = self.V(x_enc).view(B, S, self.h, self.d_h).transpose(1, 2)

        # key-query affinities
        e = queries @ torch.transpose(keys, -1, -2)  # (B, H, T, S)
        e /= self.d_h**0.5  # "scaled" part, now by d_h
        e.masked_fill_(~src_mask.unsqueeze(1).unsqueeze(1), -torch.inf)

        # attention weights
        alpha = torch.softmax(e, dim=-1)

        outputs = alpha @ values
        outputs.transpose_(1, 2)  # (B, H, T, d_h) -> (B, T, H, d_h)
        outputs = torch.reshape(outputs, (B, T, -1))
        outputs = self.W0(outputs)

        return outputs


class EncoderBlock(nn.Module):
    def __init__(self, d: int):
        super().__init__()

        # 1. attn
        self.attn = SelfAttentionMultiHead(d, h=4)

        # 2. residual + layernorm
        self.ln1 = nn.LayerNorm(d)

        # 3. ff
        d_ff = d * 4
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)

        # 4. residual + layernorm
        self.ln2 = nn.LayerNorm(d)  # just over feature dim

    def forward(self, x, src_mask):
        x1 = self.attn(x, src_mask)
        x2 = self.ln1(x + x1)

        x3 = self.ff2(torch.relu(self.ff1(x2)))

        x4 = self.ln2(x2 + x3)

        return x4


class DecoderBlock(nn.Module):
    def __init__(self, d: int):
        super().__init__()

        # 1. causal/masked self attn
        self.causal_attn = SelfAttentionMultiHead(d, h=4, masked=True)

        # 2. res + layer norm
        self.ln1 = nn.LayerNorm(d)

        # 3. multi head cross attn
        self.cross_attn = CrossAttentionMultiHead(d, h=4)

        # 4. res + layer norm
        self.ln2 = nn.LayerNorm(d)

        # 5. ff
        d_ff = d * 4
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)

        # 6. residual + layernorm
        self.ln3 = nn.LayerNorm(d)

    def forward(self, x, x_enc, src_mask, target_mask):
        x1 = self.causal_attn(x, target_mask)
        x2 = self.ln1(x + x1)

        # cross !
        x3 = self.cross_attn(x2, src_mask, x_enc)

        x4 = self.ln2(x2 + x3)

        x5 = self.ff2(torch.relu(self.ff1(x4)))

        x6 = self.ln3(x4 + x5)

        return x6


class MyTransformer(nn.Module):
    def __init__(
        self,
        embedding_size: int = 16,
        n_enc_blocks: int = 2,
        n_dec_blocks: int = 2,
        max_ctx: int = 32,
    ):
        super().__init__()
        self.embedding_size = embedding_size
        self.n_enc_blocks = n_enc_blocks
        self.n_dec_blocks = n_dec_blocks
        self.max_ctx = max_ctx

        # emb
        self.emb = nn.Embedding(num_embeddings=VOCAB_SIZE, embedding_dim=embedding_size)

        # positional encoding
        self.register_buffer("pos_enc", self._build_pos_enc())

        # enc
        self.enc_blocks = nn.ModuleList([])
        for _ in range(n_enc_blocks):
            self.enc_blocks.append(EncoderBlock(d=self.embedding_size))

        # dec
        self.dec_blocks = nn.ModuleList([])
        for _ in range(n_dec_blocks):
            self.dec_blocks.append(DecoderBlock(d=self.embedding_size))

        # final layer
        self.ff = nn.Linear(embedding_size, VOCAB_SIZE)

    def _build_pos_enc(self):
        # build out (n, d) matrix with just row index in each col (numerator)
        r = torch.arange(1, self.max_ctx + 1)
        pos_enc = torch.vstack([r for _ in range(self.embedding_size)]).T

        # the denominator, 10K comes from the orginal paper
        col = 10_000 ** (
            2 * (torch.arange(0, self.embedding_size) // 2) / self.embedding_size
        )

        # form the fraction within each trig function
        pos_enc = pos_enc / col.repeat(self.max_ctx).view(
            self.max_ctx, self.embedding_size
        )

        # apply to alternating columms
        pos_enc[:, ::2].sin_()
        pos_enc[:, 1::2].cos_()

        return pos_enc

    def forward(self, x, x_lens, y):
        """
        - x: (B, S) integer character IDs, right-padded with PAD; no source SOS/EOS.
        - x_lens: (B,) unpadded source lengths, retained for continuity with stage 4.
        - y: (B, 12) containing <YYYY-MM-DD>. All targets have the same length.
        """
        decoder_input = y[:, :-1]  # drop last token (>)
        
        B, S = x.shape
        _, T = decoder_input.shape


        # establish masks from x and y here
        src_mask = x != c2i[PAD]
        target_mask = decoder_input != c2i[PAD]

        # Encoder
        x_enc = self.emb(x) + self.pos_enc[:S, :]
        for enc in self.enc_blocks:
            x_enc = enc(x_enc, src_mask)

        # Decoder
        x_dec = self.emb(decoder_input) + self.pos_enc[:T, :]
        for dec in self.dec_blocks:
            x_dec = dec(x_dec, x_enc, src_mask, target_mask)

        out = self.ff(x_dec)

        return out

## Training time!